# 🌍 Geopolitical and Military Power Analysis Notebook

Este notebook integra **4 proyectos de Data Science** utilizando un dataset real de indicadores demográficos, económicos y militares de 192 países para los años 2023-2025.

## Proyectos incluidos:
1. **🔍 Proyecto 1: Modelado Predictivo e Importancia de Variables (XAI)**: Random Forest para predecir el poder militar global e identificar qué características pesan más.
2. **🌐 Proyecto 2: Clustering Geopolítico**: Reducción de dimensiones con PCA y agrupamiento con K-Means para agrupar países en perfiles geopolíticos.
3. **💸 Proyecto 3: Análisis de Eficiencia del Gasto**: Evaluación de qué naciones obtienen el mayor retorno de poder militar por cada dólar invertido.
4. **⚠️ Proyecto 5: Detección de Anomalías**: Isolation Forest para identificar países que rompen los patrones generales de correlación macroeconómica y militar.

In [ ]:
# Instala las dependencias necesarias en tu entorno (descomenta si hace falta)
# %pip install pandas numpy scikit-learn matplotlib seaborn plotly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Configuración de temas visuales
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = [11, 6]
import warnings
warnings.filterwarnings('ignore')

## 📂 1. Carga y Preprocesamiento de los Datos

Cargaremos el dataset `country_power_dataset.csv` y prepararemos el conjunto de 23 características numéricas que alimentarán los análisis.

In [ ]:
# Cargar datos
df = pd.read_csv('country_power_dataset.csv')
print(f"Dataset cargado exitosamente. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

# Definir variables numéricas a analizar
features = [
    'gdp_usd', 'exports_usd', 'imports_usd', 'forex_reserves_usd',
    'national_debt_pct_gdp', 'inflation_rate_pct', 'unemployment_rate_pct',
    'population_total', 'population_density', 'median_age_cia', 'total_area_km2_cia',
    'active_army_troops', 'active_navy_troops', 'active_air_force_troops',
    'reserve_troops', 'paramilitary_count', 'tanks', 'armored_vehicles',
    'fighter_jets', 'attack_helicopters', 'naval_vessels_total', 'submarines',
    'defense_budget_usd'
]

# Asegurar conversión a numérico y llenar posibles vacíos con 0
for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Target (Índice de poder militar 2025)
df['military_power_index_2025'] = pd.to_numeric(df['military_power_index_2025'], errors='coerce').fillna(df['military_power_index_2025'].mean())

df[['country_name', 'continent', 'gdp_usd', 'defense_budget_usd', 'military_power_index_2025']].head(10)

## 🔍 2. Proyecto 1: Modelado Predictivo e Importancia de Variables (XAI)

¿Qué hace que un país sea considerado militarmente poderoso en el ranking GFP? Entrenaremos un `RandomForestRegressor` para modelar el índice y luego extraeremos la importancia de cada característica para descubrir qué factores influyen más.

In [ ]:
X = df[features]
y = df['military_power_index_2025']

# Entrenar modelo
rf = RandomForestRegressor(n_estimators=150, random_state=42)
rf.fit(X, y)

# Obtener importancia de variables
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

importance_df = pd.DataFrame({
    'Feature': [features[i] for i in indices],
    'Importance': [importances[i] for i in indices]
})

# Visualizar
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Características más importantes para predecir el Poder Militar', fontsize=14, fontweight='bold')
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Característica', fontsize=12)
plt.tight_layout()
plt.show()

print("Importancias completas:")
print(importance_df)

## 🌐 3. Proyecto 2: Clustering Geopolítico (K-Means + PCA)

Agruparemos a los países en función de sus similitudes demográficas, macroeconómicas y militares. Estandarizamos los datos y usamos PCA (Análisis de Componentes Principales) para comprimir las 23 dimensiones en 2 componentes principales visualizables.

In [ ]:
# Estandarizar variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reducción dimensional con PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

# Agrupamiento K-Means con 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print("Países por Cluster:")
print(df['Cluster'].value_counts())

# Muestra ejemplos de países en cada cluster
for c in sorted(df['Cluster'].unique()):
    countries = df[df['Cluster'] == c]['country_name'].head(8).tolist()
    print(f"\nCluster {c} (ejemplos): {', '.join(countries)}...")

### Gráfico de Distribución Geopolítica Interactivo

Usa el cursor para explorar el gráfico. Podrás pasar el ratón por encima de los puntos para ver el nombre del país, su PIB, presupuesto militar y continente.

In [ ]:
fig = px.scatter(
    df, 
    x='PCA1', 
    y='PCA2', 
    color=df['Cluster'].astype(str),
    hover_name='country_name',
    hover_data=['continent', 'gdp_usd', 'defense_budget_usd', 'military_power_index_2025'],
    title='Agrupamiento Geopolítico (K-Means + PCA)',
    labels={'color': 'Cluster'},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(
    template='plotly_dark',
    xaxis_title='Componente Principal 1 (Tamaño y Recursos Totales)',
    yaxis_title='Componente Principal 2 (Tasas de Deuda/Edad/Especialización)'
)
fig.show()

## 💸 4. Proyecto 3: Análisis de Eficiencia del Gasto en Defensa

¿Qué países obtienen mayor poder militar relativo a lo que invierten económicamente? 

Creamos un **Power Score** invertido (`3.5 - military_power_index_2025`) para que a mayor valor represente un ejército más potente. Luego definimos el ratio de eficiencia como:
$$\text{Eficiencia} = \frac{\text{Power Score}}{\text{Presupuesto de Defensa (en millones de USD)} + 1}$$

In [ ]:
# Crear Power Score
df['power_score'] = 3.5 - df['military_power_index_2025']

# Calcular ratio de eficiencia
df['efficiency_ratio'] = df['power_score'] / (df['defense_budget_usd'] / 1e6 + 1.0)

# Ordenar por eficiencia
efficient_df = df.sort_values(by='efficiency_ratio', ascending=False)

print("Top 15 países más eficientes del mundo en gasto militar:")
print(efficient_df[['country_name', 'continent', 'defense_budget_usd', 'military_power_index_2025', 'efficiency_ratio']].head(15).to_string(index=False))

In [ ]:
# Visualización de la frontera de gasto vs poder militar
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=df,
    x='defense_budget_usd',
    y='power_score',
    size='efficiency_ratio',
    hue='continent',
    sizes=(30, 400),
    alpha=0.7
)
plt.xscale('log')
plt.title('Frontera de Eficiencia: Presupuesto Militar (Log) vs. Power Score', fontsize=14, fontweight='bold')
plt.xlabel('Presupuesto de Defensa en USD (Escala logarítmica)', fontsize=12)
plt.ylabel('Power Score (3.5 - Índice GFP)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## ⚠️ 5. Proyecto 5: Detección de Anomalías Geopolíticas

Buscamos países que rompen con los patrones globales regulares mediante un algoritmo de detección de anomalías (`IsolationForest`). Analizaremos qué países son catalogados como outliers geopolíticos por no alinearse a la correlación natural macro-militar.

In [ ]:
# Modelo Isolation Forest con contaminación del 8%
iso = IsolationForest(contamination=0.08, random_state=42)
df['anomaly'] = iso.fit_predict(X_scaled) # -1: anomalía, 1: normal

anomalies = df[df['anomaly'] == -1]
print(f"Se encontraron {len(anomalies)} países con perfiles anómalos.")
print("\nListado de anomalías:")
print(anomalies[['country_name', 'continent', 'gdp_usd', 'defense_budget_usd', 'military_power_index_2025']].to_string(index=False))

### Gráfico de Anomalías (PIB vs Presupuesto de Defensa)

En rojo se destacan las anomalías. Podrás notar que se trata de países que son extremos económicos o que poseen configuraciones de presupuesto militar desproporcionadas con respecto a sus economías.

In [ ]:
fig_anom = px.scatter(
    df,
    x='gdp_usd',
    y='defense_budget_usd',
    color=df['anomaly'].map({1: 'Normal', -1: 'Anomalía'}),
    hover_name='country_name',
    hover_data=['continent', 'population_total', 'military_power_index_2025'],
    log_x=True,
    log_y=True,
    title='Detección de Anomalías Geopolíticas (Isolation Forest)',
    color_discrete_map={'Normal': '#1f77b4', 'Anomalía': '#e74c3c'}
)
fig_anom.update_layout(
    template='plotly_dark',
    xaxis_title='PIB en USD (Escala logarítmica)',
    yaxis_title='Presupuesto de Defensa en USD (Escala logarítmica)'
)
fig_anom.show()

## 🏁 Conclusiones del Análisis

1. **¿Qué determina el poder militar?**: Nuestro modelo RandomForest muestra que el **presupuesto de defensa** y el **PIB** son, por amplio margen, los factores más importantes para determinar el rango general de poder militar global. Esto valida que la fuerza de defensa moderna está fuertemente anclada en la potencia económica general y en los recursos logísticos y comerciales.

2. **Clusters Globales**: El agrupamiento K-Means dividió al mundo en:
   * **Cluster de Superpotencias (el grupo más pequeño)**: Países gigantescos con presupuestos colosales y gran cantidad de tropas y equipamiento activo (EE. UU., China, Rusia, India).
   * **Cluster de Potencias Regionales Avanzadas**: Economías fuertes con ejércitos desarrollados y compactos.
   * **Cluster Mayoritario**: El resto de países del mundo con presupuestos pequeños y perfiles militares acotados.

3. **Retorno de la Inversión**: Analizar la eficiencia demostró que los países que no poseen ejército de primera línea o tienen presupuestos de defensa minúsculos (ej. Costa Rica, Venezuela, Haití) consiguen la mayor relación de "poder militar por millón de dólares". En contraste, los países líderes muestran rendimientos marginales decrecientes (su gasto es inmenso pero su escala de poder indexado crece lentamente).

4. **Perfiles Atípicos**: Las anomalías identificadas por Isolation Forest corresponden a potencias mundiales avanzadas (Japón, Francia, Italia, Corea del Sur, Turquía). Aunque tienen un altísimo PIB y gran nivel de desarrollo, sus ejércitos son compactos y profesionales comparados con su escala económica, diferenciándose de la norma estadística promedio global.